In [1]:
import os
import sys
import pandas as pd
from datetime import datetime
sys.path.append(os.path.abspath("../../"))
from src.unitelma_processor import UnitelmaProcessor

import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

ROOT_DIR = os.getenv("ROOT_DIR")

/Volumes/T7/documents/github/dropout-prediction/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TRAIN_FILE = os.path.join(ROOT_DIR, ".data_unitelma/train_test/train_timeseries.csv")
TEST_FILE = os.path.join(ROOT_DIR, ".data_unitelma/train_test/test_timeseries.csv")
OUTPUT_DIR = os.path.join(ROOT_DIR, "notebooks_unitelma/runs/outputs_run_01")
os.makedirs(OUTPUT_DIR, exist_ok=True)

processor = UnitelmaProcessor(TRAIN_FILE, TEST_FILE, OUTPUT_DIR, False)

[INFO] Device PyTorch -> mps


In [4]:
LAGS = [7, 14]
results = []

for l in LAGS:
    X_train, y_train, X_test, y_test, features = processor.prepare_dl_data(lag=l)

    #X_train, X_test, _ = processor.apply_feature_selection(X_train, y_train, X_test)

    print(f"[TRAIN] ...")
    metrics, best_model = processor.evaluate_dl_model(X_train, y_train, X_test, y_test, input_dim=97)
    
    metrics['lag'] = l
    metrics.pop('best_params', None)
    results.append(metrics)

        

if results:
    df_results = pd.DataFrame(results)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    csv_path = os.path.join(OUTPUT_DIR, f"metrics_ml_{timestamp}.csv")
    
    df_results.to_csv(csv_path, index=False)
    print(f"\n[SUCCESS] Complete -> {csv_path}")
    
    # Mostriamo la tabella riassuntiva a video nel notebook
    display(df_results)

[TRAIN] ...
[TRAIN] ...

[SUCCESS] Complete -> /Volumes/T7/documents/github/dropout-prediction/notebooks_unitelma/runs/outputs_run_01/metrics_ml_20260531_220054.csv


,accuracy,precision,recall,f1,roc_auc,pr_auc,lag
0,0.5678,0.7976,0.5678,0.6051,0.7252,0.7906,7
1,0.6250,0.7845,0.6250,0.6612,0.7107,0.7784,14
